In [ ]:
library(dreamlet)
library(zenith)
library(scater)

# 01 Preprocess data

In [ ]:
sce = schard::h5ad2sce('../02_combination_10x_pb/hdf5/20250811_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.annot_corrected.index_dedup.cg.h5ad') # change path/name as necessary

In [ ]:
# check if X is counts or not; if it is rename to counts
if (all(assay(sce, "X") == floor(assay(sce, "X")))) {
  assayNames(sce)[assayNames(sce) == "X"] <- "counts"
  cat("Assay 'X' renamed to 'counts' (contains integer values)\n")
} else {
  cat("Assay 'X' are not counts; check your input\n")
}

In [ ]:
sce

In [ ]:
keep_cells <- !is.na(colData(sce)$vireo_assignment) & 
              colData(sce)$vireo_assignment != "unassigned" &
             !is.na(colData(sce)$primary_diagnosis)
sce <- sce[, keep_cells]

In [ ]:
# keep only one batch per donor, in case donor present in both parsebio and 10x
meta <- as.data.frame(colData(sce))
meta$cell_id <- rownames(meta)  # Add cell barcodes as a column

# Find the batch with the most cells for each donor
keep_cells <- unlist(
  lapply(split(meta, meta$vireo_assignment), function(df) {
    tab <- table(df$batch)
    best_batch <- names(tab)[which.max(tab)]
    df$cell_id[df$batch == best_batch]
  })
)

# Subset the SCE object to keep only those cells
sce <- sce[, keep_cells]

In [ ]:
# adjust diagnosis labels
colData(sce)$primary_diagnosis[colData(sce)$primary_diagnosis == "Other neurological disorder"] <- "Healthy Control"
colData(sce)$primary_diagnosis[colData(sce)$primary_diagnosis == "Healthy Control"] <- "Control"
colData(sce)$primary_diagnosis[colData(sce)$primary_diagnosis == "Idiopathic PD"] <- "PD"

In [ ]:
# # add PMI metadata, correct path as necessary
# meta <- read.csv("/donor_statistics/crn_metadata/upload/CLINPATH.csv", stringsAsFactors = FALSE)

# # Ensure column names match: subject_id and duration_PMI
# colData(sce)$duration_PMI <- meta$duration_PMI[
#   match(colData(sce)$vireo_assignment, meta$subject_id)
# ]


In [ ]:
sce <- sce[rowSums(counts(sce) > 0) > 0, ]

# compute QC metrics
qc <- perCellQCMetrics(sce)

# remove cells with few or many detected genes
ol <- isOutlier(metric = qc$detected, nmads = 2, log = TRUE)
#sce <- sce[, !ol]

# set variable indicating PD  or control
sce$PD <- sce$primary_diagnosis

In [ ]:
table(ol)

# 02 Aggregate to pseudobulk

In [ ]:
# create unique identifier for each sample
sce$id <- paste0(sce$batch, sce$vireo_assignment)

# Create pseudobulk data by specifying cluster_id and sample_id
pb <- aggregateToPseudoBulk(sce,
  assay = "counts",
  cluster_id = "consensus_cell_type_corrected",
  sample_id = "id",
  verbose = FALSE
)

# one 'assay' per cell type
assayNames(pb)

In [ ]:
pb

# 03 Voom for pseudobulk

Apply voom-style normalization for pseudobulk counts within each cell cluster using voomWithDreamWeights to handle random effects (if specified).

In [ ]:
colnames(pb)

In [ ]:
pb

In [ ]:
# add PMI metadata, correct path as necessary
meta <- read.csv("/crn_metadata/upload/CLINPATH.csv", stringsAsFactors = FALSE)

meta$subject_id <- trimws(as.character(meta$subject_id))
colData(pb)$vireo_assignment <- trimws(as.character(colData(pb)$vireo_assignment))

# Create index mapping from vireo_assignment to subject_id
idx <- match(colData(pb)$vireo_assignment, meta$subject_id)

# Map duration_PMI using the index
colData(pb)$duration_pmi <- meta$duration_pmi[idx]

In [ ]:
colData(pb)

In [ ]:
# Normalize and apply voom/voomWithDreamWeights
form = ~ age_at_collection + sex + biobank_name + batch + duration_pmi
res.proc <- processAssays(pb, form, min.count = 5, min.cells = 5)


res.proc

In [ ]:
details(res.proc)

In [ ]:
details(res.proc)$n_genes

In [ ]:
# show voom plot for each cell clusters
plotVoom(res.proc)

# 04 Variance partitioning

In [ ]:
# run variance partitioning analysis
form <- ~ age_at_collection + sex + biobank_name + batch + duration_pmi + PD
vp.lst <- fitVarPart(res.proc, form)

In [ ]:
# Show variance fractions at the gene-level for each cell type
genes <- vp.lst$gene[2:15]
plotPercentBars(vp.lst[vp.lst$gene %in% genes, ])

In [ ]:
# Summarize variance fractions genome-wide for each cell type
options(repr.plot.width = 20, repr.plot.height = 20)
plotVarPart(vp.lst, label.angle = 60)

# 05 Differential expression

In [ ]:
# Differential expression analysis within each assay,
# evaluated on the voom normalized data
form <- ~ age_at_collection + sex + biobank_name + batch + duration_pmi + PD
res.dl <- dreamlet(res.proc, form)

# names of estimated coefficients
coefNames(res.dl)

In [ ]:
# the resulting object of class dreamletResult
# stores results and other information
res.dl

## 051 Volcano plot

Red is FDR < 0.05

In [ ]:
plotVolcano(res.dl, coef = "PDPD")

In [ ]:
library(dplyr)
topTable(res.dl, coef = "PDPD", number=Inf) %>% write.table(file="dreamlet/2026_fdp_ASAP_cc_all_bulk_PDPD.tsv", quote=FALSE, sep="\t", row.names=FALSE)

In [ ]:
topTable(res.dl, coef = "PDPD", number = Inf) %>%
  as.data.frame() %>%
  filter(adj.P.Val <= 0.05) %>%
  count(assay)

In [ ]:
sessionInfo()

R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: Rocky Linux 9.6 (Blue Onyx)

Matrix products: default
BLAS/LAPACK: /envs/dreamlet/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.utf8       LC_NUMERIC=C             
 [3] LC_TIME=en_US.utf8        LC_COLLATE=C             
 [5] LC_MONETARY=en_US.utf8    LC_MESSAGES=en_US.utf8   
 [7] LC_PAPER=en_US.utf8       LC_NAME=C                
 [9] LC_ADDRESS=C              LC_TELEPHONE=C           
[11] LC_MEASUREMENT=en_US.utf8 LC_IDENTIFICATION=C      

time zone: Europe/Brussels
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] GO.db_3.20.0                org.Hs.eg.db_3.20.0        
 [3] AnnotationDbi_1.68.0        dplyr_1.1.4                
 [5] scater_1.34.1               scuttle_1.16.0             
 [7] zenith_1.8.0                dreamlet_1.6.0             
 [9] SingleCellExperiment_1.28.0 SummarizedExperiment_1.36.0
[11] Biobase_2.66.0              GenomicRanges_1.58.0       
[13] GenomeInfoDb_1.42.0         IRanges_2.40.0             
[15] S4Vectors_0.44.0            BiocGenerics_0.52.0        
[17] MatrixGenerics_1.18.0       matrixStats_1.5.0          
[19] variancePartition_1.36.2    BiocParallel_1.40.0        
[21] limma_3.62.1                ggplot2_3.5.2              

loaded via a namespace (and not attached):
  [1] splines_4.4.3             pbdZMQ_0.3-14            
  [3] filelock_1.0.3            bitops_1.0-9             
  [5] tibble_3.3.0              graph_1.84.0             
  [7] XML_3.99-0.17             lifecycle_1.0.4          
  [9] Rdpack_2.6.4              mixsqp_0.3-54            
 [11] edgeR_4.4.0               lattice_0.22-7           
 [13] MASS_7.3-64               backports_1.5.0          
 [15] magrittr_2.0.3            metafor_4.8-0            
 [17] DBI_1.2.3                 minqa_1.2.8              
 [19] RColorBrewer_1.1-3        abind_1.4-5              
 [21] zlibbioc_1.52.0           EnvStats_3.1.0           
 [23] purrr_1.1.0               rmeta_3.0                
 [25] msigdbr_25.1.1            RCurl_1.98-1.17          
 [27] GenomeInfoDbData_1.2.13   ggrepel_0.9.6            
 [29] pbkrtest_0.5.5            irlba_2.3.5.1            
 [31] annotate_1.84.0           DelayedMatrixStats_1.28.0
 [33] codetools_0.2-20          DelayedArray_0.32.0      
 [35] tidyselect_1.2.1          UCSC.utils_1.2.0         
 [37] farver_2.1.2              viridis_0.6.5            
 [39] ScaledMatrix_1.14.0       lme4_1.1-37              
 [41] BiocFileCache_2.14.0      base64enc_0.1-3          
 [43] mathjaxr_1.8-0            jsonlite_2.0.0           
 [45] BiocNeighbors_2.0.0       iterators_1.0.14         
 [47] systemfonts_1.2.3         tools_4.4.3              
 [49] progress_1.2.3            ragg_1.4.0               
 [51] Rcpp_1.1.0                glue_1.8.0               
 [53] gridExtra_2.3             SparseArray_1.6.0        
 [55] IRdisplay_1.1             schard_0.0.1             
 [57] withr_3.0.2               numDeriv_2016.8-1.1      
 [59] fastmap_1.2.0             rhdf5filters_1.18.0      
 [61] boot_1.3-31               rsvd_1.0.5               
 [63] caTools_1.18.3            digest_0.6.37            
 [65] truncnorm_1.0-9           R6_2.6.1                 
 [67] textshaping_1.0.1         colorspace_2.1-1         
 [69] Cairo_1.6-2               scattermore_1.2          
 [71] gtools_3.9.5              RSQLite_2.4.2            
 [73] RhpcBLASctl_0.23-42       tidyr_1.3.1              
 [75] generics_0.1.4            data.table_1.17.8        
 [77] corpcor_1.6.10            prettyunits_1.2.0        
 [79] httr_1.4.7                S4Arrays_1.6.0           
 [81] pkgconfig_2.0.3           gtable_0.3.6             
 [83] blob_1.2.4                XVector_0.46.0           
 [85] remaCor_0.0.18            htmltools_0.5.8.1        
 [87] GSEABase_1.68.0           scales_1.4.0             
 [89] png_0.1-8                 fANCOVA_0.6-1            
 [91] reformulas_0.4.1          ashr_2.2-63              
 [93] reshape2_1.4.4            uuid_1.2-1               
 [95] nlme_3.1-168              curl_6.4.0               
 [97] nloptr_2.2.1              rhdf5_2.50.0             
 [99] repr_1.1.7                cachem_1.1.0             
[101] stringr_1.5.1             KernSmooth_2.23-26       
[103] parallel_4.4.3            vipor_0.4.7              
[105] metadat_1.4-0             RcppZiggurat_0.1.8       
[107] pillar_1.11.0             grid_4.4.3               
[109] vctrs_0.6.5               gplots_3.2.0             
[111] BiocSingular_1.22.0       dbplyr_2.5.0             
[113] beachmat_2.22.0           mashr_0.2.79             
[115] xtable_1.8-4              beeswarm_0.4.0           
[117] Rgraphviz_2.50.0          evaluate_1.0.4           
[119] KEGGgraph_1.66.0          invgamma_1.2             
[121] mvtnorm_1.3-3             cli_3.6.5                
[123] locfit_1.5-9.12           compiler_4.4.3           
[125] rlang_1.1.6               crayon_1.5.3             
[127] SQUAREM_2021.1            labeling_0.4.3           
[129] plyr_1.8.9                ggbeeswarm_0.7.2         
[131] stringi_1.8.7             viridisLite_0.4.2        
[133] assertthat_0.2.1          babelgene_22.9           
[135] lmerTest_3.1-3            Biostrings_2.74.0        
[137] aod_1.3.3                 Matrix_1.7-3             
[139] IRkernel_1.3.2            hms_1.1.3                
[141] sparseMatrixStats_1.18.0  bit64_4.6.0-1            
[143] Rhdf5lib_1.28.0           KEGGREST_1.46.0          
[145] statmod_1.5.0             rbibutils_2.3            
[147] Rfast_2.1.0               broom_1.0.9              
[149] memoise_2.0.1             RcppParallel_5.1.9       
[151] bit_4.6.0                 EnrichmentBrowser_2.36.0
```

```